In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local db")
else:
    print("using AWS db")

using AWS db


In [5]:
from datetime import datetime

from birddog.database import Database
from birddog.database_dashboard import (
    get_doc_id_by_process_code,
    assign_docs_to_pages,
    split_doc_list_by_code,
)

In [6]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label="Elapsed"):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.3f}s")

In [7]:
db = Database()

2026-05-04 12:14:03,451 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     0.16    39.00       0.00           24


In [8]:
with timer():
    dids = get_doc_id_by_process_code(db, ["P1", "P2", "FX"])

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
43098
Elapsed: 35.722s


In [9]:
len(dids)

43098

In [10]:
all_dids = list(dids.keys())

In [11]:
with timer():
    dm, pm = assign_docs_to_pages(db, all_dids[:500])

2026-05-04 12:14:39,644 [INFO] _make_doc_map: loading doc records (500)
2026-05-04 12:14:41,455 [INFO] _make_page_tree: loading owning page records (502)
2026-05-04 12:14:43,780 [INFO] _make_page_tree: loading parent page records (15)
2026-05-04 12:14:43,989 [INFO] _make_page_tree: loading parent page records (10)
2026-05-04 12:14:44,218 [INFO] _make_page_tree: loading parent page records (6)
Elapsed: 4.831s


In [12]:
sum=0
for p, v in pm.items():
    #if v['level'] not in ('case', 'volume'):
    if v['level'] == 'opus':
        if not v.get('assigned_parent') and v.get('assigned_docs', []):
            print(f"{p}: {v['label']} - no upward assigned doc")
        sum += len(v.get('assigned_docs', []))
        print(f"{p}: {v['label']} ({v.get('level', '')}), assigned parent={v.get('assigned_parent')}, num_docs={len(v.get('assigned_docs', []))}")
sum

49311: DADNO-R/R-6508/22d (opus), assigned parent=26511, num_docs=1
27075: DADNO-R/R-6508/7 (opus), assigned parent=26511, num_docs=477
30211: DAOO-D/39/5 (opus), assigned parent=28529, num_docs=1
44873: DADNO-R/R-6508/43 (opus), assigned parent=26511, num_docs=1
263275: TSDAVO-_/1/5 (opus), assigned parent=264993, num_docs=3
262571: TSDAVO-_/1/3 (opus), assigned parent=264993, num_docs=2
222460: DAPO-R/R-2034/3 (opus), assigned parent=219858, num_docs=1
43894: DADNO-R/R-2492/2 (opus), assigned parent=42273, num_docs=1
32022: DADNO-R/R-6508/17d (opus), assigned parent=31915, num_docs=1
38360: DADNO-R/R-2859/1 (opus), assigned parent=49487, num_docs=1
260217: TSDAVO-_/1/6 (opus), assigned parent=264993, num_docs=2
37850: DADNO-R/R-3807/1 (opus), assigned parent=44810, num_docs=1
40924: DADNO-R/R-2874/1 (opus), assigned parent=42519, num_docs=1
42334: DADNO-R/R-2492/1 (opus), assigned parent=42273, num_docs=1
32639: DAOO-R/R-8085/1 (opus), assigned parent=32631, num_docs=6
31915: DADNO-R

501

In [13]:
#pm = split_doc_assignments_by_code(pm, dids)

In [14]:
#for p, v in pm.items():
    #if v['level'] not in ('case', 'volume'):
    #if v['level'] == 'opus':
        #if not v.get('assigned_parent') and v.get('assigned_docs', []):
        #    print(f"{p}: {v['label']} - no upward assigned doc")
        #sum += len(v.get('assigned_docs', []))
        #assigned_docs = v.get("assigned_docs", {})
        #if assigned_docs:
        #    print(f"{p}: {v['label']} ({v.get('level', '')}), assigned parent={v.get('assigned_parent')}")
        #    for k, l in assigned_docs.items():
        #        print(f"    {k}: {len(l)}")

In [15]:
def _scan_opus_summary(db, codes):
    fields = [ f"{code}_documents" for code in codes ]
    fields.extend(["page", "label"])
    print(fields)

    cursor = None
    result = []
    while True:
        records, cursor = db.scan("OpusSummary", cursor=cursor, limit=500, fields=fields)
        result.extend(records)
        if not records or not cursor:
            return result

In [18]:
def update_opus_summary(page_map, doc_code_map):
    codes = { code for code in doc_code_map.values() }

    opus_summary_map = {}
    opus_summary_records_to_delete = []
    for rec in _scan_opus_summary(db, codes):
        linked_page_id = rec.get("page", {}).get("Id")
        # each summary record must uniquely link to a valid page
        if linked_page_id and linked_page_id not in opus_summary_map:
            opus_summary_map[linked_page_id] = rec
        else:
            opus_summary_records_to_delete.append(rec)
    print(opus_summary_map)

    opus_summary_records_to_create = set()
    opus_summary_records_to_update = set()

    for page_id, page in page_map.items():
        #if v['level'] not in ('case', 'volume'):
        if page['level'] == 'opus':
            print(
                f"{page_id}: {page['label']} ({page.get('level', '')}),"
                f" assigned parent={page.get('assigned_parent')},"
                f" num_docs={len(page.get('assigned_docs', []))}")
            summary_record = opus_summary_map.get(page_id)
            if summary_record:
                # known opus page - check if update needed
                assigned_docs = split_doc_list_by_code(
                    page.get('assigned_docs', []), doc_code_map)
                for code, doc_list in assigned_docs:
                    doc_link_field = f"{code}_documents"
                    summary_doc_list = summary_rec.get(doc_link_field, [])
                    if set(doc_list) != set(summary_doc_list):
                        summary_rec[doc_link_field] = doc_list
                        opus_summary_records_to_update.add(page_id)
            else:
                # new page to be added
                opus_summary_records_to_create.add(page_id)

    if opus_summary_records_to_create:
        records = [{ "label": page_map[page_id]["label"]} for page_id in opus_summary_records_to_create]
        print("create:", records)
        record_ids = db.write("OpusSummary", records)
        for rec_id, page_id in zip(record_ids, opus_summary_records_to_create):
            assigned_docs = split_doc_list_by_code(
                page_map[page_id].get('assigned_docs', []), doc_code_map)
            for code, doc_ids in assigned_docs.items():
                if doc_ids:
                    doc_link_field = f"{code}_documents" 
                    db.create_links("OpusSummary", doc_link_field, rec_id, doc_ids)

    return opus_summary_map, opus_summary_records_to_create, opus_summary_records_to_update, opus_summary_records_to_delete 
                

In [19]:
update_opus_summary(pm, dids)

['P2_documents', 'FX_documents', 'P1_documents', 'page', 'label']
2026-05-04 14:26:27,273 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   26.00     0.00    39.00       0.00           24
{29388: {'Id': 423, 'label': 'DAOO-D/1/2', 'page': {'Id': 29388, 'title': 'ДАОО/1/2'}, 'P1_documents': [], 'P2_documents': [], 'FX_documents': []}}
49311: DADNO-R/R-6508/22d (opus), assigned parent=26511, num_docs=1
27075: DADNO-R/R-6508/7 (opus), assigned parent=26511, num_docs=477
30211: DAOO-D/39/5 (opus), assigned parent=28529, num_docs=1
44873: DADNO-R/R-6508/43 (opus), assigned parent=26511, num_docs=1
263275: TSDAVO-_/1/5 (opus), assigned parent=264993, num_docs=3
262571: TSDAVO-_/1/3 (opus), assigned parent=264993, num_docs=2
222460: DAPO-R/R-2034/3 (opus), assigned parent=219858, num_docs=1
43

KeyError: 'Id'

In [ ]:
x = set(range(3))

In [ ]:
x.insert(10)

In [ ]:
x.add(10)

In [ ]:
x